# ggplot2 (grammar of graphics)

A refresher on **ggplot2** — the dominant R plotting package (part of the tidyverse) that builds graphics from a *grammar*: you declaratively map data columns to visual channels (x, y, colour, size) and stack **layers** rather than issuing imperative draw commands. Once the grammar clicks, the same handful of verbs compose into almost any statistical graphic.

**Domain:** Data Analysis & Research  ·  **runnable:** no — R package, not a Python library  ·  _R_

> This is an **R** notebook. The snippets below run in an R session (REPL, `Rscript`, RStudio, Quarto, or an R Jupyter kernel) — **not** this Python kernel. Results are described in prose; there is no fabricated cell output.

## 1. What & Why

**What it is.** `ggplot2` (Hadley Wickham) is an R package that implements Leland Wilkinson's *Grammar of Graphics*. Instead of "draw a bar chart" or "draw a scatter plot," you describe a plot as a combination of independent pieces: a **dataset**, **aesthetic mappings** (which column drives x, y, colour, …), one or more **geometric layers** (`geom_point`, `geom_line`, `geom_bar`, …), optional **statistical transformations**, a **coordinate system**, and **facets**. Adding `+` layers and components composes the final graphic.

**The problem it solves.** Base R graphics and most plotting APIs are *imperative*: you draw, then tweak, then add a legend by hand, then realise the colours don't match. ggplot2 is *declarative* — you state the mapping from data to visuals once, and it handles scales, legends, and guides automatically and consistently. Because the pieces are orthogonal, you change one facet (swap a `geom`, add a facet grid, log-scale an axis) without rewriting the rest.

**Reach for it when:** you work in R, your data is a tidy data frame (one row per observation, one column per variable), and you want publication-quality statistical graphics — especially anything involving grouping, faceting (small multiples), or mapping a third/fourth variable onto colour and shape.

**Don't reach for it when:** you live in Python (use `matplotlib`/`seaborn`/`plotly`, or `plotnine` for a ggplot clone), you need interactive/web dashboards (use `plotly`, or `ggplotly()` to wrap a ggplot), your data is wide/untidy and reshaping is more work than it's worth, or you need 3D / GPU-scale rendering.

## 2. Mental Model

**A ggplot is a stack of transparent layers over a shared coordinate grid. The data and the aesthetic mapping say *where* each row lands on the grid; each `geom` is a sheet that draws those positions a different way; scales translate data values into pixels/colours; facets photocopy the whole stack once per subgroup.**

Read a plot specification as a sentence with slots:

```
ggplot(data, aes(x = ?, y = ?, colour = ?))   # the mapping: data columns -> visual channels
  + geom_*()        # the layer(s): how to render the mapped points
  + stat_*()        # optional statistical transform (binning, smoothing, summarising)
  + scale_*()       # how data values map to pixels / colours / sizes
  + facet_*()       # split into small multiples by a grouping variable
  + coord_*()       # the coordinate system (cartesian, flipped, polar, map)
  + theme_*()       # non-data ink: fonts, gridlines, background
```

Two ideas make the grammar pay off:

1. **Mapping vs setting.** `aes(colour = species)` *maps* a column to colour (so the colour varies by data and a legend appears). `colour = "blue"` *outside* `aes()` *sets* a constant. Confusing these two is the #1 beginner mistake.
2. **Inheritance.** Aesthetics declared in the top-level `ggplot(aes(...))` are inherited by every layer; a `geom` can override or add its own. Put the shared mapping at the top, the layer-specific bits in the geom.

## 3. Key Concepts

- **Data.** A tidy data frame (or tibble): one row per observation, one column per variable. ggplot2 expects **long** format — to plot three measurements as three coloured lines, you need a `variable`/`value` pair, not three wide columns. Reshape first with `tidyr::pivot_longer()`.
- **Aesthetics (`aes`).** The mapping from data columns to visual channels: `x`, `y`, `colour`, `fill`, `shape`, `size`, `alpha`, `linetype`, `group`. **Inside `aes()` = mapped (varies by data, gets a legend); outside `aes()` = set (a constant).**
- **Geoms.** Geometric objects = one visual layer each: `geom_point`, `geom_line`, `geom_col`/`geom_bar`, `geom_boxplot`, `geom_histogram`, `geom_smooth`, `geom_tile`, `geom_text`. You stack as many as you like.
- **Stats.** Statistical transforms that compute new values before drawing. `geom_bar` secretly runs `stat_count`; `geom_histogram` runs `stat_bin`; `geom_smooth` runs `stat_smooth` (loess/lm). Every geom has a default stat and vice-versa. Use `stat = "identity"` (or `geom_col`) when your data already holds the bar heights.
- **Scales.** Control how data values become aesthetics: `scale_x_log10()`, `scale_colour_brewer()`, `scale_fill_viridis_c()`, `scale_y_continuous(labels = scales::comma)`. Every mapped aesthetic has a scale; you only touch it to override defaults.
- **Facets.** Small multiples: `facet_wrap(~ var)` lays panels out in a grid that wraps; `facet_grid(row_var ~ col_var)` crosses two variables into a true matrix of panels.
- **Coordinates.** `coord_cartesian(xlim = …)` (zoom *without* dropping data — see Gotchas), `coord_flip()` (swap axes), `coord_polar()`, `coord_sf()` (maps).
- **Themes.** All non-data ink: `theme_minimal()`, `theme_bw()`, plus `theme(legend.position = "bottom", …)` for fine control. `labs(title=, x=, y=, colour=)` sets titles and axis/legend labels.
- **`+` not `%>%`.** You *add* plot components with `+`. The tidyverse pipe `%>%`/`|>` chains data transformations *before* the plot; inside a ggplot it's `+`. Mixing them up is a common error.

## 4. Setup

ggplot2 is an **R package** installed from CRAN — there is no `pip install`. You need the R interpreter first, then the package (usually via the whole tidyverse).

```bash
# 1. Install R itself (the interpreter):
brew install --cask r                 # macOS;  or download from https://cran.r-project.org
sudo apt-get install r-base           # Debian/Ubuntu
```

```r
# 2. Inside an R session, install from CRAN:
install.packages("ggplot2")           # just the plotting package
install.packages("tidyverse")         # recommended: ggplot2 + dplyr + tidyr + readr + ...

# Handy companions:
install.packages(c("scales",          # axis label formatters (comma, percent, dollar)
                   "patchwork",        # compose multiple ggplots into one figure
                   "ggrepel",          # non-overlapping text labels
                   "viridis"))         # colourblind-safe palettes

library(ggplot2)                       # or library(tidyverse) to load the whole stack
packageVersion("ggplot2")              # confirm install
```

Ways to actually run the snippets below:

```bash
Rscript plot.R                         # run a script; ggsave() writes the file (see Example 4)
R -q                                   # interactive REPL
```

- **RStudio / Positron** — plots render in the Plots pane interactively.
- **Quarto / R Markdown** — a ```` ```{r} ```` chunk renders the figure inline (see the `quarto` notebook).
- **Jupyter with the IRkernel** (`install.packages("IRkernel"); IRkernel::installspec()`) — gives a real R kernel so ggplots render in a notebook. This notebook uses the *Python* kernel, so it stays conceptual.

## 5. Worked Examples

**These run in an R session, not this Python kernel.** Paste into the R REPL, save as `plot.R` and run `Rscript plot.R`, or use an R Jupyter kernel. Each example uses a dataset that ships with R/ggplot2 (`mpg`, `diamonds`, `economics`) so they run with no downloads. Expected results are described in prose — there is no fabricated cell output.

### Example 1 — Scatter with a third variable, grouping, and a smoother

`mpg`: fuel economy of 234 cars. Map engine size to x, highway mpg to y, drivetrain to colour, then overlay a per-group trend line. This is the canonical ggplot "hello world" and shows mapping, layering, and an automatic legend.

```r
library(ggplot2)

ggplot(mpg, aes(x = displ, y = hwy, colour = drv)) +
  geom_point(alpha = 0.7) +                       # layer 1: the raw points (alpha is SET, constant)
  geom_smooth(method = "lm", se = TRUE) +         # layer 2: one linear fit PER colour group
  scale_colour_brewer(palette = "Set1") +         # nicer, colourblind-friendlier palette
  labs(
    title = "Bigger engines, worse highway mileage",
    x = "Engine displacement (L)", y = "Highway MPG", colour = "Drivetrain"
  ) +
  theme_minimal()
```

**What you get / what to notice:**
- A downward scatter; because `colour = drv` is *inside* `aes()`, points are coloured by drivetrain (4/f/r) and a legend appears automatically.
- `geom_smooth` inherits the `colour` mapping, so you get **three** regression lines (one per group), each with a shaded 95% confidence ribbon.
- `alpha = 0.7` sits *outside* `aes()` → every point is 70% opaque (a constant), not mapped to data.
- Swap `geom_smooth(method = "lm")` for the default `geom_smooth()` to get a wiggly loess curve instead of straight lines.

### Example 2 — Bars: `geom_bar` (counts) vs `geom_col` (values), and faceting

The single most common bar-chart confusion: does ggplot *count rows for you*, or do you already have the heights? `geom_bar` runs `stat_count` (counts rows per category); `geom_col` uses your values as-is (`stat = "identity"`).

```r
library(ggplot2)
library(dplyr)

# (a) geom_bar — let ggplot COUNT rows: how many cars of each class?
ggplot(mpg, aes(x = class, fill = drv)) +
  geom_bar(position = "dodge") +              # side-by-side bars; "stack" (default) or "fill" (100%)
  labs(y = "count of cars")

# (b) geom_col — you SUPPLY the heights (here, a precomputed summary)
mpg %>%
  group_by(class) %>%
  summarise(mean_hwy = mean(hwy)) %>%
  ggplot(aes(x = reorder(class, mean_hwy), y = mean_hwy)) +   # reorder bars by value!
  geom_col(fill = "steelblue") +
  coord_flip() +                              # horizontal bars — easier to read long labels
  labs(x = NULL, y = "mean highway MPG")

# (c) Faceting — one panel per drivetrain, shared axes
ggplot(mpg, aes(x = displ, y = hwy)) +
  geom_point() +
  facet_wrap(~ drv, nrow = 1) +               # small multiples; facet_grid(year ~ drv) crosses two vars
  theme_bw()
```

**What to notice:**
- (a) bars are *counts*; `fill = drv` splits each bar by drivetrain. `position = "dodge"` puts them side by side; `"fill"` would normalise each bar to 100% (good for proportions).
- (b) you had to compute `mean_hwy` yourself, so use `geom_col`. `reorder(class, mean_hwy)` sorts bars by height — ggplot otherwise orders factors alphabetically, which almost no one wants.
- (c) `facet_wrap(~ drv)` gives three panels sharing the same x/y scales, so they're directly comparable. Add `scales = "free_y"` only if per-panel ranges differ wildly.

### Example 3 — Distributions and reshaping wide → long

Two everyday tasks: comparing a distribution across groups, and the pivot you almost always need first. ggplot wants **long** data — to plot several columns as several series, melt them into a `name`/`value` pair.

```r
library(ggplot2)
library(tidyr)

# Distribution of price by cut — boxplot + jittered points, log y-axis
ggplot(diamonds, aes(x = cut, y = price, fill = cut)) +
  geom_boxplot(outlier.alpha = 0.2) +
  scale_y_log10(labels = scales::dollar) +     # price is heavily right-skewed -> log scale
  labs(x = NULL, y = "price (log scale)") +
  theme_minimal() +
  theme(legend.position = "none")              # fill is redundant with x, so drop the legend

# WIDE -> LONG so each measurement becomes its own coloured density
# Suppose `df` has columns: id, height, weight, bmi (three measures, wide)
df_long <- pivot_longer(df, cols = c(height, weight, bmi),
                        names_to = "measure", values_to = "value")

ggplot(df_long, aes(x = value, fill = measure)) +
  geom_density(alpha = 0.4) +                  # overlapping semi-transparent densities
  facet_wrap(~ measure, scales = "free") +     # each measure on its own scale
  theme_bw()
```

**What to notice:**
- `scale_y_log10()` rescales the axis cleanly and `scales::dollar` formats ticks as `$1,000` — far better than manual axis hacking.
- The wide→long pivot is the unlock: with three separate columns you'd need three `geom_*` calls and a manual legend; with one long column ggplot maps `measure` to fill and builds the legend for free.
- `facet_wrap(~ measure, scales = "free")` lets each density use its own range (height in cm and bmi in kg/m² shouldn't share an axis).

### Example 4 — Time series, saving to file, and composing plots

Real workflows end in a *file*, not a screen — and often combine several panels. Use `ggsave()` for reproducible output and `patchwork` to lay plots side by side.

```r
library(ggplot2)
library(patchwork)

p1 <- ggplot(economics, aes(x = date, y = unemploy / 1000)) +
  geom_line(colour = "firebrick") +
  labs(title = "US unemployment", x = NULL, y = "millions") +
  theme_minimal()

p2 <- ggplot(economics, aes(x = date, y = psavert)) +
  geom_area(fill = "steelblue", alpha = 0.5) +
  labs(title = "Personal savings rate", x = NULL, y = "%") +
  theme_minimal()

# patchwork composes ggplots with + (side by side) or / (stacked):
combined <- p1 / p2                            # p1 on top, p2 below

# ggsave: reproducible export — set dimensions and DPI explicitly, no screenshotting.
ggsave("report.png", combined, width = 8, height = 6, dpi = 300)
ggsave("report.pdf", combined, width = 8, height = 6)   # vector output for print
```

**What to notice:**
- A ggplot is a normal R object — assign it (`p1 <-`), print it later, or modify it (`p1 + theme_bw()`). Nothing is drawn until the object is printed or saved.
- `ggsave()` saves the **last printed plot** by default, or a named plot if you pass it. Always set `width`/`height`/`dpi` so the figure is reproducible and the text isn't microscopic — the on-screen size is not what gets written.
- `patchwork`'s `+`, `/`, and `|` operators arrange multiple ggplots into one figure; `plot_layout()` controls relative sizes and a shared legend.

## 6. Gotchas & Pitfalls

- **`aes()` vs constant — the cardinal sin.** `geom_point(aes(colour = "red"))` does **not** make points red — it maps every point to a *categorical value* literally named `"red"`, picks ggplot's first default colour (salmon), and adds a useless legend. To set a constant colour, put it *outside* `aes()`: `geom_point(colour = "red")`.
- **`+` vs `%>%`.** Build plots with `+`; pipe data transformations with `%>%`/`|>`. `ggplot(d) %>% geom_point()` errors; `ggplot(d) + geom_point()` is right. You *can* pipe data **into** `ggplot()` and then switch to `+`.
- **`geom_bar` vs `geom_col`.** `geom_bar` counts rows (`stat_count`); if you already have heights and use `geom_bar(aes(y = value))` you'll get an error or a wrong plot. Use `geom_col` (or `geom_bar(stat = "identity")`) for precomputed values.
- **Factor order is alphabetical.** ggplot orders categorical axes and legends by factor levels, which default to alphabetical. Reorder explicitly with `reorder(x, value)`, `forcats::fct_reorder()`, or by setting factor `levels` — otherwise your "Low/Medium/High" axis reads "High/Low/Medium".
- **`xlim()`/`ylim()` silently delete data.** `scale_x_continuous(limits = …)`, `xlim()`, and `ylim()` *drop* rows outside the range **before** stats run — so a `geom_smooth` or boxplot is computed on the wrong subset. To *zoom* without dropping data, use `coord_cartesian(xlim = …, ylim = …)`.
- **Overplotting hides the data.** Thousands of points become one black blob. Use `alpha`, `geom_jitter`/`position_jitter`, `geom_hex`, `geom_bin2d`, or `geom_density_2d` to reveal density.
- **Continuous vs discrete colour scales.** Mapping a numeric column to `colour` gives a continuous gradient; a factor gives discrete hues. If your "category" is stored as a number (e.g. `cyl`), wrap it: `aes(colour = factor(cyl))`.
- **Nothing renders.** A ggplot only draws when *printed*. Inside a `for` loop, function, or script you must explicitly `print(p)` (or `ggsave`) — assigning to a variable alone shows nothing.
- **Legends, titles, and label text.** Override default labels via `labs()` and the relevant `scale_*(name = …)`; the legend title comes from the aesthetic name unless you rename it.

## 7. When to Use vs Alternatives

| Option | Best at | Weaknesses vs ggplot2 |
|---|---|---|
| **ggplot2 (R)** | Declarative statistical graphics, faceting/small-multiples, consistent legends & scales, publication-quality static figures, huge extension ecosystem | R-only; static by default; steep-ish initial grammar; very large datasets can be slow to render |
| **base R graphics** | Quick throwaway plots, total low-level control, no dependencies | Imperative and verbose; you manage legends/scales by hand; hard to compose or theme consistently |
| **plotnine (Python)** | The ggplot grammar *in Python* — nearly identical API for pandas users | Smaller ecosystem/feature lag vs R ggplot2; less mature |
| **matplotlib / seaborn (Python)** | The Python default; seaborn adds tidy statistical defaults on top | Imperative (matplotlib); seaborn is less composable than the full grammar |
| **plotly / ggplotly** | Interactive, zoomable, web/dashboard plots; `ggplotly()` wraps an existing ggplot | Heavier output; fine static-print control is weaker |
| **D3.js / Vega-Lite** | Bespoke, fully interactive web visualisations | Far more work; not for quick analysis; Vega-Lite is the closest "grammar" analogue |

**Rule of thumb:** if you're in R and the data is tidy, ggplot2 is the default — reach past it only for interactivity (`plotly`/`ggplotly`), throwaway one-liners (base R), or another language entirely. In Python, `plotnine` gives you the same grammar; otherwise `seaborn`. See the `seaborn`, `plotly`, `matplotlib`, and `r-language` notebooks for the neighbours.

## 8. Resources

- **ggplot2 official site & function reference** — every geom/stat/scale with examples: https://ggplot2.tidyverse.org/
- **"ggplot2: Elegant Graphics for Data Analysis"** (Wickham, Navarro, Pedersen) — the canonical book, free online: https://ggplot2-book.org/
- **"R for Data Science" (2e), Data Visualisation chapter** — the best gentle on-ramp, free online: https://r4ds.hadley.nz/data-visualize
- **The R Graph Gallery** — hundreds of copy-paste ggplot recipes by chart type: https://r-graph-gallery.com/
- **Posit ggplot2 cheatsheet (PDF)** — the one-page grammar reference worth pinning: https://rstudio.github.io/cheatsheets/data-visualization.pdf
- **patchwork** (composing plots) and **scales** (axis formatters) docs: https://patchwork.data-imaginist.com/ · https://scales.r-lib.org/

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE